In [1]:
import pandas as pd
import duckdb

In [2]:
pd.set_option('display.max_columns', None)

In [4]:
conn = duckdb.connect(r"..\data\cms_mpd.duckdb", read_only=True)

In [33]:
conn.execute("select * from information_schema.tables limit 5").fetchdf() #.to_csv("../data/db_metadata.csv")

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action,TABLE_COMMENT
0,cms_mpd,bronze,basic_formulary,BASE TABLE,None,None,None,None,None,YES,NO,None,None
1,cms_mpd,bronze,beneficiary_cost,BASE TABLE,None,None,None,None,None,YES,NO,None,None
2,cms_mpd,bronze,excluded_drugs,BASE TABLE,None,None,None,None,None,YES,NO,None,None
3,cms_mpd,bronze,geographic_locator,BASE TABLE,None,None,None,None,None,YES,NO,None,None
4,cms_mpd,bronze,indication_coverage,BASE TABLE,None,None,None,None,None,YES,NO,None,None


In [40]:
conn.execute("""select  *
                        from bronze.pde_sample

             
              limit 5""").fetchdf()

,PDE_ID,BENE_ID,SRVC_DT,PD_DT,PRSCRBR_ID_QLFYR_CD,PRSCRBR_ID,RX_SRVC_RFRNC_NUM,PROD_SRVC_ID,PLAN_CNTRCT_REC_ID,PLAN_PBP_REC_NUM,CMPND_CD,DAW_PROD_SLCTN_CD,QTY_DSPNSD_NUM,DAYS_SUPLY_NUM,FILL_NUM,DSPNSNG_STUS_CD,DRUG_CVRG_STUS_CD,ADJSTMT_DLTN_CD,NSTD_FRMT_CD,PRCNG_EXCPTN_CD,CTSTRPHC_CVRG_CD,GDC_BLW_OOPT_AMT,GDC_ABV_OOPT_AMT,PTNT_PAY_AMT,OTHR_TROOP_AMT,LICS_AMT,PLRO_AMT,CVRD_D_PLAN_PD_AMT,NCVRD_PLAN_PD_AMT,TOT_RX_CST_AMT,RX_ORGN_CD,RPTD_GAP_DSCNT_NUM,BRND_GNRC_CD,PHRMCY_SRVC_TYPE_CD,PTNT_RSDNC_CD,SUBMSN_CLR_CD,source_file,snapshot_quarter,load_ts
0,-10602819806,-10000010286527,16-Feb-2022,16-Feb-2022,01,9999995869,-10602819806,54569854000,Z0001,999,0,2,21,21,1,None,C,None,None,None,None,0.00,0,0.00,0,0,0.23,0.93,0.00,1.16,3,0,G,06,01,None,pde.csv,2025-Q3,2026-04-09 18:47:24.474896+07:00
1,-10602819807,-10000010286527,16-Feb-2022,16-Feb-2022,01,9999995869,-10602819807,63629764004,Z0001,999,0,0,21,21,1,None,C,None,None,None,None,0.00,0,0.00,0,0,0.38,1.50,0.00,1.88,4,0,B,07,01,None,pde.csv,2025-Q3,2026-04-09 18:47:24.474896+07:00
2,-10602819808,-10000010286527,16-Feb-2022,16-Feb-2022,01,9999995869,-10602819808,55111058810,Z0001,999,0,3,21,21,1,None,C,None,None,None,None,0.00,0,0.00,0,0,4.03,16.10,0.00,20.13,4,0,G,07,01,None,pde.csv,2025-Q3,2026-04-09 18:47:24.474896+07:00
3,-10602819809,-10000010286527,09-Mar-2022,09-Mar-2022,01,9999977729,-10602819809,67544036931,Z0001,999,0,0,7,7,1,None,C,None,None,None,None,0.00,0,0.00,0,0,0.28,1.12,0.00,1.40,4,0,B,06,01,None,pde.csv,2025-Q3,2026-04-09 18:47:24.474896+07:00
4,-10602819810,-10000010286527,09-Mar-2022,09-Mar-2022,01,9999977729,-10602819810,68788762301,Z0001,999,0,6,7,7,1,None,C,None,None,None,None,0.00,0,0.00,0,0,0.40,1.60,0.00,2.00,3,0,G,03,01,None,pde.csv,2025-Q3,2026-04-09 18:47:24.474896+07:00


In [ ]:

import sys, json, time
from pathlib import Path
sys.path.insert(0, 'D:/STUDY/PRACTICE/cms-mpd-research/sandbox/src')
from cms_mpd import PipelineConfig, build_training_dataset
config = PipelineConfig(project_root=Path('D:/STUDY/PRACTICE/cms-mpd-research/sandbox'))
start = time.time()
path = build_training_dataset(config=config)
elapsed = round(time.time() - start, 2)
print(json.dumps({
  'dataset_path': str(path),
  'metadata_path': str(config.training_dataset_metadata_path),
  'elapsed_seconds': elapsed
}, indent=2))

In [23]:


im_con = duckdb.connect()

In [24]:
im_con.execute("""
        create temp view train_temp_view as 
               select * 
               from read_csv_auto('../data/training/2025-Q3/full/hybrid_reranker_dataset.csv')
""")

In [32]:
im_con.execute("""select 
                    *
               --scenario_id, count(*) as cnt
                from train_temp_view
               where 1=1
               --and scenario_bundle = 'insulin_only'
                and scenario_id = 'insulin_only_01008'
               group by all
               """).fetchdf()

,scenario_id,scenario_bundle,plan_key,plan_name,current_rules_rank,current_rules_score,fit_score,cost_score,premium_score,coverage_score,access_score,stability_score,annual_premium,annual_drug_oop,annual_total_cost,coverage_status,requested_drug_count,covered_drug_count_request,covered_drug_share,priced_drug_share,uncovered_drug_count,uncovered_drug_share,priced_drug_count,restriction_count,deductible_exposure_total,initial_coverage_oop_total,lis_adjusted_oop_total,negotiated_price_total,oop_cap_savings_total,excluded_drug_count,excluded_drug_share,missing_price_drug_count,missing_price_drug_share,channel_unavailable_count,channel_unavailable_share,mail_order_dependency_flag,mail_order_dependency_share,monthly_drug_oop_variance,monthly_total_variance,channel_switch_count,insulin_risk_flag,insulin_nonpreferred_dependency_count,insulin_nonpreferred_dependency_share,approximate_match_count,exact_match_count,preferred_match_count,network_flag,network_risk_score,scenario_profile,fallback_group,lis_status,age_band,pharmacy_preference,zip_density_category,zipcode_density_score,beneficiary_chronic_condition_count,nearest_preferred_distance_bucket,in_area_pharmacies,preferred_retail_count,preferred_mail_count,covered_drug_count,insulin_drug_count,pa_rate,st_rate,ql_rate,excluded_rate,deductible,served_counties,contract_year,benefit_design,simulation_policy,feature_version,match_review_required_flag,unknown_network_data_flag,unsafe_reason_count,candidate_plan_count_service_area,candidate_plan_count_ranked,plans_with_unknown_network_count,plans_dropped_due_to_missing_data,weak_label_score,heuristic_score,weak_label_rank,weak_label_relevance
0,insulin_only_01008,insulin_only,S2893001000,Blue MedicareRx Value Plus (PDP),47.0,-905.20,59.00,74.00,74.00,68.0,40.0,100.0,595.2,0.00,595.20,partial,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,unknown,1.5,insulin_chronic,network_unknown,partial,65-74,auto,rural,0.0,1.0,3.0,0.0,0.0,0.0,3167.0,24.0,0.3228,0.0050,0.4077,0.0,590.0,41.0,2025.0,2025_redesign,cost_realism_v1,research_v4,0.0,1.0,2.0,57.0,57.0,15.0,0.0,-1780.4,-1715.4,46,0
1,insulin_only_01008,insulin_only,S5884102000,Humana Basic Rx Plan (PDP),52.0,-1540.00,59.00,46.28,46.28,68.0,40.0,100.0,1230.0,0.00,1230.00,partial,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,unknown,1.5,insulin_chronic,network_unknown,partial,65-74,auto,rural,0.0,1.0,3.0,0.0,0.0,0.0,3042.0,19.0,0.2594,0.0062,0.4753,0.0,590.0,41.0,2025.0,2025_redesign,cost_realism_v1,research_v4,0.0,1.0,2.0,57.0,57.0,15.0,0.0,-3050.0,-2985.0,52,0
2,insulin_only_01008,insulin_only,H8578012000,Health New England Medicare Value (HMO),26.0,-250.00,59.00,100.00,100.00,68.0,92.0,100.0,0.0,0.00,0.00,partial,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,adequate,0.0,insulin_chronic,never_local_coverable,partial,65-74,auto,rural,0.0,1.0,1.0,142.0,72.0,11.0,3411.0,21.0,0.2497,0.0108,0.4426,0.0,490.0,4.0,2025.0,2025_redesign,cost_realism_v1,research_v4,0.0,0.0,1.0,57.0,57.0,15.0,0.0,-500.0,-450.0,26,0
3,insulin_only_01008,insulin_only,H2230017000,Medicare PPO Blue SaverRx (PPO),30.0,-250.00,59.00,100.00,100.00,68.0,92.0,100.0,0.0,0.00,0.00,partial,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,adequate,0.0,insulin_chronic,never_local_coverable,partial,65-74,auto,rural,0.0,1.0,1.0,1022.0,576.0,0.0,3303.0,26.0,0.3130,0.0072,0.4031,0.0,0.0,11.0,2025.0,2025_redesign,cost_realism_v1,research_v4,0.0,0.0,1.0,57.0,57.0,15.0,0.0,-500.0,-450.0,30,0
4,insulin_only_01008,insulin_only,H5216250000,HumanaChoice H5216-250 (PPO),42.0,-357.20,59.00,97.06,97.06,68.0,52.0,100.0,67.2,0.00,67.20,partial,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0

In [22]:
duckdb.execute("""select scenario_bundle, count(*) as cnt
                from train_view""").fetchdf()

BinderException: Binder Error: column "scenario_bundle" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(scenario_bundle)" if the exact value of "scenario_bundle" is not important.

LINE 1: select scenario_bundle, count(*) as cnt
               ^

In [17]:
train_df = pd.read_csv("../data\\training\\2025-Q3\\full\\hybrid_reranker_dataset.csv")
train_df.head()

,scenario_id,scenario_bundle,plan_key,plan_name,current_rules_rank,current_rules_score,fit_score,cost_score,premium_score,coverage_score,access_score,stability_score,annual_premium,annual_drug_oop,annual_total_cost,coverage_status,requested_drug_count,covered_drug_count_request,covered_drug_share,priced_drug_share,uncovered_drug_count,uncovered_drug_share,priced_drug_count,restriction_count,deductible_exposure_total,initial_coverage_oop_total,lis_adjusted_oop_total,negotiated_price_total,oop_cap_savings_total,excluded_drug_count,excluded_drug_share,missing_price_drug_count,missing_price_drug_share,channel_unavailable_count,channel_unavailable_share,mail_order_dependency_flag,mail_order_dependency_share,monthly_drug_oop_variance,monthly_total_variance,channel_switch_count,insulin_risk_flag,insulin_nonpreferred_dependency_count,insulin_nonpreferred_dependency_share,approximate_match_count,exact_match_count,preferred_match_count,network_flag,network_risk_score,scenario_profile,fallback_group,lis_status,age_band,pharmacy_preference,zip_density_category,zipcode_density_score,beneficiary_chronic_condition_count,nearest_preferred_distance_bucket,in_area_pharmacies,preferred_retail_count,preferred_mail_count,covered_drug_count,insulin_drug_count,pa_rate,st_rate,ql_rate,excluded_rate,deductible,served_counties,contract_year,benefit_design,simulation_policy,feature_version,match_review_required_flag,unknown_network_data_flag,unsafe_reason_count,candidate_plan_count_service_area,candidate_plan_count_ranked,plans_with_unknown_network_count,plans_dropped_due_to_missing_data,weak_label_score,heuristic_score,weak_label_rank,weak_label_relevance
0,insulin_only_01001,insulin_only,H2256028000,Tufts Medicare Preferred HMO Saver Rx (HMO),1.0,10000.0,100.00,100.00,100.00,100.0,100.0,100.0,0.0,0.0,0.0,full,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3851.52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,adequate,0.0,insulin_chronic,full_coverage,partial,65-74,auto,urban,2.0,1.0,0.0,968.0,627.0,0.0,3554.0,37.0,0.1933,0.0086,0.0955,0.0,0.0,10.0,2025.0,2025_redesign,cost_realism_v1,research_v4,0.0,0.0,0.0,57.0,57.0,15.0,0.0,10955.0,10500.0,1,5
1,insulin_only_01001,insulin_only,H2256046000,Tufts Medicare Preferred HMO Smart Saver Rx (HMO),2.0,10000.0,100.00,100.00,100.00,100.0,100.0,100.0,0.0,0.0,0.0,full,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3851.52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,adequate,0.0,insulin_chronic,full_coverage,partial,65-74,auto,urban,2.0,1.0,0.0,968.0,627.0,0.0,3554.0,37.0,0.1933,0.0086,0.0955,0.0,0.0,10.0,2025.0,2025_redesign,cost_realism_v1,research_v4,0.0,0.0,0.0,57.0,57.0,15.0,0.0,10955.0,10500.0,2,4
2,insulin_only_01001,insulin_only,H2256026003,Tufts Medicare Preferred HMO Basic Rx (HMO),3.0,9869.2,99.26,94.29,94.29,100.0,100.0,100.0,130.8,0.0,130.8,full,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3811.26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,adequate,0.0,insulin_chronic,full_coverage,partial,65-74,auto,urban,2.0,1.0,0.0,104.0,63.0,0.0,3554.0,37.0,0.1933,0.0086,0.0955,0.0,0.0,2.0,2025.0,2025_redesign,cost_realism_v1,research_v4,0.0,0.0,0.0,57.0,57.0,15.0,0.0,10693.4,10238.4,12,0
3,insulin_only_01001,insulin_only,H2256018008,Tufts Medicare Preferred HMO Value Rx (HMO),4.0,9377.2,96.46,72.80,72.80,100.0,100.0,100.0,622.8,0.0,622.8,full,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3811.26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,adequate,0.0,insulin_chronic,full_coverage,partial,65-74,auto,urban,2.0,1.0,0.0,104.0,63.0,0.0,3554.0,37.0,0.1933,0.0086,0.0955,0.0,0.0,2.0,2025.0,2025_redesign,cost_realism_v1,research_v4,0.0,0.0,0.0,57.0,57.0,15.0,0.0,9709.4,9254.4,20,0
4,insulin_only_01001,insulin_only,H0137001000,CCA One Care (Medicare-Medicaid Plan),5.0,9960.0,94.24,100.00,100.00,100.0,52.0,100.0,0.0,0.0,0.0,full,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3697.11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0

In [19]:
train_df.groupby("scenario_bundle").size()

scenario_bundle
insulin_only              2686
insulin_plus_chronic      2686
low_generic               2686
maintenance_brand         2686
rural_access_sensitive    2686
specialty_high_cost       2686
dtype: int64

In [41]:
conn.close()